# Haystack 2.x RAG Demo

Haystack 2.x uses a **Pipeline** architecture with explicit component connections.

**Why Haystack:**
- Type-safe component connections
- Built-in production patterns
- Excellent hybrid search support

**Prerequisites:**
```bash
pip install haystack-ai sentence-transformers
ollama pull qwen3:4b
```

In [1]:
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import HuggingFaceLocalGenerator
import requests

# Sample documents
documents = [
    Document(content="Remote work is allowed 3 days per week with manager approval.", meta={"topic": "hr"}),
    Document(content="Vacation policy: 25 days annual leave for full-time employees.", meta={"topic": "hr"}),
    Document(content="Code reviews are mandatory before merging to main branch.", meta={"topic": "eng"}),
    Document(content="Performance reviews occur quarterly with written feedback.", meta={"topic": "hr"}),
]

print(f"✓ {len(documents)} documents ready")

✓ 4 documents ready


---

## 1. Build Indexing Pipeline

In [2]:
# Create document store
document_store = InMemoryDocumentStore()

# Create embedder
doc_embedder = SentenceTransformersDocumentEmbedder(model="all-MiniLM-L6-v2")
doc_embedder.warm_up()

# Embed and store documents
docs_with_embeddings = doc_embedder.run(documents)
document_store.write_documents(docs_with_embeddings["documents"])

print(f"✓ Indexed {document_store.count_documents()} documents")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Indexed 4 documents


---

## 2. Build Retrieval Pipeline

In [3]:
# Create retrieval pipeline
retrieval_pipeline = Pipeline()

# Add components
text_embedder = SentenceTransformersTextEmbedder(model="all-MiniLM-L6-v2")
retriever = InMemoryEmbeddingRetriever(document_store=document_store, top_k=3)

retrieval_pipeline.add_component("text_embedder", text_embedder)
retrieval_pipeline.add_component("retriever", retriever)

# Connect components
retrieval_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")

print("✓ Retrieval pipeline built")

# Test retrieval
query = "How many vacation days do I get?"
result = retrieval_pipeline.run({"text_embedder": {"text": query}})

print(f"\nQuery: \"{query}\"")
print("-" * 50)
for doc in result["retriever"]["documents"]:
    print(f"  [{doc.score:.3f}] {doc.content}")

✓ Retrieval pipeline built


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: "How many vacation days do I get?"
--------------------------------------------------
  [0.645] Vacation policy: 25 days annual leave for full-time employees.
  [0.318] Remote work is allowed 3 days per week with manager approval.
  [0.065] Performance reviews occur quarterly with written feedback.


---

## 3. Full RAG Pipeline (with Ollama)

In [4]:
# Custom Ollama generator (Haystack 2.x style)
from haystack import component

@component
class OllamaGenerator:
    def __init__(self, model: str = "qwen3:4b"):
        self.model = model
    
    @component.output_types(replies=list)
    def run(self, prompt: str):
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": self.model, "prompt": prompt, "stream": False}
        )
        return {"replies": [response.json().get("response", "")]}

# Build full RAG pipeline
rag_pipeline = Pipeline()

rag_pipeline.add_component("text_embedder", SentenceTransformersTextEmbedder(model="all-MiniLM-L6-v2"))
rag_pipeline.add_component("retriever", InMemoryEmbeddingRetriever(document_store=document_store, top_k=3))
rag_pipeline.add_component("prompt_builder", PromptBuilder(
    template="""Answer based only on the context. No thinking, just answer.

Context:
{% for doc in documents %}
- {{ doc.content }}
{% endfor %}

Question: {{ query }}
Answer:"""
))
rag_pipeline.add_component("generator", OllamaGenerator())

# Connect components
rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "generator")

print("✓ Full RAG pipeline built")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✓ Full RAG pipeline built


In [5]:
# Run the RAG pipeline
try:
    query = "What is the vacation policy?"
    result = rag_pipeline.run({
        "text_embedder": {"text": query},
        "prompt_builder": {"query": query}
    })
    
    print(f"Query: \"{query}\"")
    print("=" * 60)
    print(f"Answer: {result['generator']['replies'][0][:200]}...")
except Exception as e:
    print(f"RAG pipeline error: {e}")
    print("Make sure Ollama is running with qwen3:4b")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "What is the vacation policy?"
Answer: 25 days annual leave for full-time employees...


---

## Haystack 2.x Key Concepts

**Components:** Self-contained units with `run()` method  
**Pipelines:** Connect components via input/output sockets  
**Document Stores:** Pluggable storage (In-Memory, Qdrant, Elasticsearch)

**Advantages over LangChain:**
- Type-safe connections (fails fast on mismatch)
- Cleaner serialization
- Better debugging (each component is inspectable)

**Production pattern:** Export pipeline as YAML for deployment.